In [ ]:
# 1. COLAB & TERMINAL GUIDE: To send a command to the Linux terminal running behind Colab, we put an exclamation mark (!) at the start of the line.
# In a real Linux terminal, these commands would NOT have the exclamation mark.

# LINUX TERMINAL COMMAND: 'wget' (Web Get) is the basic tool for downloading a file from the internet directly to a server.

#    PURPOSE OF THIS CELL (DEMONSTRATION): This is how you would download the FULL raw dataset in a real project.
#    The links below are the full data files for our paper (Illumina ~441 MB, Nanopore ~2.1 GB).

print("Downloading the FULL raw data...\n")


# Short-read (Illumina, paired-end):
!wget -P full_data_example "http://ftp.sra.ebi.ac.uk/vol1/fastq/SRR136/036/SRR13680736/SRR13680736_1.fastq.gz"
!wget -P full_data_example "http://ftp.sra.ebi.ac.uk/vol1/fastq/SRR136/036/SRR13680736/SRR13680736_2.fastq.gz"

# Long-read (Nanopore, PacBio single-end):
!wget -P full_data_example "https://sra-pub-run-odp.s3.amazonaws.com/sra/SRR13680735/SRR13680735"


In [ ]:
# 2. COLAB & TERMINAL GUIDE: we use an exclamation mark (!) to call Linux's package manager (apt-get) through Colab.
# But when writing these commands on a real Linux terminal, there is NO exclamation mark (!) (just sudo apt-get...).

# TOOL PARAMETER: the '-y' (yes) parameter next to the 'install' command automatically confirms the installation.

!sudo apt-get update
!sudo apt-get install fastqc -y

In [ ]:
# 3. COLAB & TERMINAL GUIDE: we put an exclamation mark (!) at the start of the folder-creation (mkdir) and analysis-triggering (fastqc) commands
# to run them in the Colab terminal. No exclamation mark is used in a real terminal.

# GUIDE: the 'mkdir' (Make Directory) command creates a folder. The '-p' (parents) parameter is a very critical
# safety measure in bioinformatics workflows; it stops the command from erroring out and halting the whole analysis if the folder already exists.
!mkdir -p qc_reports

# TOOL PARAMETERS AND LOGIC:
# 1. FastQC can process several files side by side at the same time (in parallel). Just leave a space between the file paths.
# 2. The '-o' (Output) parameter tells FastQC, "Which folder should I write the .html (visual report) and .zip (raw statistics) files to?"

# APPLY: manually type the actual names and extensions of the paired-end short-read files you saw in the
# left-panel file explorer below, first 1 (Forward) then 2 (Reverse), with ONE SPACE between them!

!fastqc /content/full_data_example/SRR13680736_1.fastq.gz /content/full_data_example/SRR13680736_2.fastq.gz -o qc_reports/

In [ ]:
# 4. COLAB & TERMINAL GUIDE: NanoPlot is a Python library. We put an exclamation mark (!) at the start of the line to run
# the Python package manager 'pip' command via Colab. You'll write it without the exclamation mark on a real terminal.

print("Installing the NanoPlot tool and all required Python libraries...")
!pip install NanoPlot
print("\nGreat! Our NanoPlot installation completed successfully.")

In [ ]:
# 5. COLAB & TERMINAL GUIDE: we use an exclamation mark (!) to start the NanoPlot analysis via Colab. No exclamation mark on a real terminal.

# TOOL PARAMETERS AND LOGIC:
# 1. The '--fastq' parameter tells the tool the format of the file it will analyze.
# 2. The '-o' (Output) parameter sets the destination folder for the results.
# 3. The '--threads 2' parameter speeds up the process by telling the tool "use 2 processor cores of the computer at the same time for this analysis."

# APPLY: manually type the actual name and extension (.fastq or .fastq.gz) of the long-read (Nanopore) file
# you saw in the left-panel file explorer below!

!NanoPlot --fastq /content/full_data_example/SRR13680735 -o qc_reports/nanoplot_result --threads 2

In [ ]:
# 6. ERROR DIAGNOSIS: Finding Out the File's Real Format
print("Analyzing the file's real identity...\n")

!file /content/full_data_example/SRR13680735

### FIXING THE ERROR: Converting an SRA Archive to FASTQ Format (Extraction)

The raw `SRR` files we download from NCBI are compressed binary files in the SRA (Sequence Read Archive) format. For our bioinformatics tools (NanoPlot, Filtlong, etc.) to process this data, we need to use the `fasterq-dump` tool from the `SRA-Toolkit` package to open this archive and convert the data into the universal `.fastq` text format.

In [ ]:
# 7. FIX: Converting SRA to FASTQ and Re-running NanoPlot

print("1. Installing the SRA-Toolkit package...\n")
!sudo apt-get update
!sudo apt-get install sra-toolkit -y

print("\n2. Opening the SRA archive and converting it to FASTQ format (this may take a few minutes)...\n")
# The fasterq-dump command reads the SRA file and creates a .fastq text file in the same directory.
!fasterq-dump /content/full_data_example/SRR13680735 --outdir /content/full_data_example/ --threads 2

print("\n3. FASTQ conversion complete. Starting Quality Control (Pre-QC) with NanoPlot...\n")
!NanoPlot --fastq /content/full_data_example/SRR13680735.fastq -o qc_reports/nanoplot_result --threads 2

print("\nThe whole process completed successfully! Please open the 'qc_reports/nanoplot_result' folder in the file explorer and check the HTML report.")

In [ ]:
# 8. COLAB & TERMINAL GUIDE: we put an exclamation mark (!) at the start of the line to install the MultiQC tool
# through the Colab terminal interface. You'll write it without the exclamation mark on a real terminal.

print("Installing the MultiQC tool...")
!pip install multiqc
print("\nMultiQC installation completed successfully!")

In [ ]:
# 9. COLAB & TERMINAL GUIDE: we put an exclamation mark (!) at the start of the cell code to trigger MultiQC
# and have it scan the folder. No exclamation mark is used on a real terminal.

# TOOL PARAMETERS AND LOGIC:
# 1. The first 'qc_reports/' in the command tells MultiQC, "Go into this folder and find all the analysis logs you recognize (FastQC, NanoPlot, etc.)"
# 2. The 'qc_reports/' after the '-o' (Output) parameter means "Combine all this data and save the final HTML report you'll produce into this same folder too."
# Bioinformatics note: MultiQC is a very 'smart' tool. Even if you give it a folder full of thousands of unrelated files, it will only find and pick out the outputs of the bioinformatics tools it supports.

print("MultiQC is scanning all subfolders and combining the reports...")
!multiqc qc_reports/ -o qc_reports/

print("\nSuccess! Our combined interactive quality control report is ready.")
print("Please open the 'qc_reports' folder in the left panel, right-click the newly created 'multiqc_report.html' file and download it to your computer.")

## 10. Removing Noise From Short-Read Raw Data: Trimming and Filtering Workflows

We've reviewed our quality control (QC) reports and taken an "X-ray" of our raw data. Now we need to remove sequencing-induced signal noise to prepare our raw data for the alignment (mapping) step with tools like BWA-MEM.

Removing noise from raw data in bioinformatics consists of two distinct, sequential mechanical steps:
1. **Trimming:** Cutting off technical adapter sequences or bases with dropped Phred quality at the ends of a read.
2. **Filtering:** After trimming is complete, asking "Is what's left of this read still long/good enough for my analysis?" and removing reads that don't meet the criteria from the dataset entirely.

### 10.1. Short-Read Raw Data: Illumina and Ion Torrent Approaches
* **Illumina (Paired-End) Data:** Offers a high-throughput raw data profile, but with increasing substitution noise toward the ends of reads. To avoid breaking the synchronization between the forward (R1) and reverse (R2) read files, noise removal must be run simultaneously (in paired-end mode).
* **Ion Torrent (Single-End) Data:** Usually read in a single direction. Substitution errors are rare, but the raw data has a lot of deletion/insertion noise in homopolymer regions. So trimming and filtering are carried out on a single file.

### 10.2. Tool Not Used in Our Reference Paper: Cutadapt (All-in-One Solution - But We'll Use the Paper's Trimmomatic Tool Instead)
`Cutadapt` is the industry-standard tool that can do both trimming and filtering on short-read raw data at the same time, with a lot of parameter flexibility.

*(Note: we install it directly with the Python package manager, pip, to keep it fully compatible with the Colab environment.)*

In [ ]:
# 10.2. COLAB TERMINAL GUIDE: Installing Cutadapt
print("Installing the short-read noise removal tool (Cutadapt)...\n")

# Using pip so it's 100% compatible with the Colab Python environment:
!pip install cutadapt dnaio

print("\nGreat! Cutadapt installation completed successfully.")

## 10.3: Removing Noise From the Short-Read Raw Data (Our Reference Paper's Trimmomatic Workflow)

To stay 100% faithful to our reference paper's *Gluconobacter cerinus FLW-1* methodology, we'll use the **Trimmomatic** tool to remove low-quality reads and adapters from the Illumina sequences.

Trimmomatic scans the read end-to-end with a "Sliding Window" algorithm, and ruthlessly cuts the sequence the moment the quality score drops.

### 10.4. Installing the Trimmomatic Tool
We're installing this Java-based, industry-standard noise removal tool on our virtual machine.

In [ ]:
# 10.4. COLAB TERMINAL GUIDE: Installing Trimmomatic
print("Installing the short-read noise removal tool (Trimmomatic)...\n")
!sudo apt-get update
!sudo apt-get install trimmomatic -y
print("\nTrimmomatic installation completed successfully.")

### 10.5. MAIN WORKFLOW: Trimming and Filtering Illumina (Paired-End) With Trimmomatic
In this step, we cut the universal Illumina adapters (TruSeq3), trim ends where the average Phred score in a 4-base window drops below 20 using the Sliding Window logic (`SLIDINGWINDOW:4:20`), and completely filter out sequences that end up shorter than 30 bases after trimming (`MINLEN:30`), since they're no longer useful.

*(Bioinformatics note: since Trimmomatic runs in Paired-End mode, it separates reads whose pairing gets broken into separate 'unpaired' files to avoid breaking synchronization; we'll only continue our analysis with the 'filtered' - i.e. paired - ones.)*

**Parameter Deep Dive: `ILLUMINACLIP` Math**
Behind the `ILLUMINACLIP:/usr/share/trimmomatic/TruSeq3-PE.fa:2:30:10` parameter in our command lies a very precise search algorithm:
* **`TruSeq3-PE.fa`**: the file path of the universal Illumina adapter library (in FASTA format) to be removed from the data.
* **`2` (Seed Mismatch):** the maximum number of errors (tolerance) allowed when searching for the adapter sequence (in the seed match).
* **`30` (Palindrome Clip):** the accuracy threshold score that definitively triggers a cut when adapter overlap is detected between paired (Paired-End) reads.
* **`10` (Simple Clip):** the simple match score that triggers a cut when only a short fragment of the adapter is detected on a single read.

In [ ]:
# 10.5. COLAB APPLICATION: Illumina Workflow With Trimmomatic
!mkdir -p filtered_reads
print("TRIMMING AND FILTERING the ILLUMINA (Paired-End) raw data with TRIMMOMATIC...\n")

!TrimmomaticPE -threads 2 \
    /content/full_data_example/SRR13680736_1.fastq.gz /content/full_data_example/SRR13680736_2.fastq.gz \
    filtered_reads/filtered_SRR13680736_1.fastq filtered_reads/unpaired_1.fastq \
    filtered_reads/filtered_SRR13680736_2.fastq filtered_reads/unpaired_2.fastq \
    ILLUMINACLIP:/usr/share/trimmomatic/TruSeq3-PE.fa:2:30:10 \
    SLIDINGWINDOW:4:20 MINLEN:30

print("\nTrimmomatic noise removal for the Illumina data is complete. Outputs are in the 'filtered_reads/' folder.")

### 11. MAIN WORKFLOW: Quality Control of the Trimmed Illumina Reads (Post-QC)

We have to prove how successfully the Trimmomatic algorithm cut the adapters and shaved off low-quality bases (below Phred score 20). We now run the **FastQC** tool we used on the raw data on day one, this time on our cleaned paired (R1 and R2) files in the `filtered_reads` folder.

In [ ]:
# 11. COLAB APPLICATION: FastQC After Noise Removal (Post-QC)
!mkdir -p post_qc_fastqc

print("Running quality control (Post-QC) on the filtered ILLUMINA short reads...\n")

!fastqc filtered_reads/filtered_SRR13680736_1.fastq \
        filtered_reads/filtered_SRR13680736_2.fastq \
        -o post_qc_fastqc/

print("\nPost-QC analysis complete! Please open the 'post_qc_fastqc/' folder in the Colab file explorer and check the HTML reports.")
print("Comparing this to the old report, you'll see the adapter warnings have disappeared and the quality plots have risen into the green zone!")

## 12. Removing Noise From the Long-Read Raw Data (Oxford Nanopore Workflow)

Our reference paper followed a very specific and mathematically precise strategy on the Oxford Nanopore reads for assembling the *Gluconobacter cerinus FLW-1* genome. We'll apply exactly these same academic parameters in our project:

1. **Porechop (Trimming):** adapters and chimeric sequences will be removed from the data.
2. **Filtlong (Smart Filtering):** after trimming, the data will go through a three-stage filter:
   * All reads shorter than 2,000 bases (bp) will be removed.
   * The **Illumina short reads** we cleaned of noise (Lesson 7.2) **will be used as a 16-mer reference**, letting us smartly select the highest-quality 90% of the Nanopore reads.
   * To avoid overloading the hybrid assembly algorithm, the data pool will be reduced to a 1,500-megabase (Mbp) subset.

### 12.1. Installing the Long-Read Tools
We're installing the trimming and filtering tools needed for this job split on the virtual machine.

In [ ]:
# 12.1. COLAB TERMINAL GUIDE: Installing Porechop and Filtlong
print("Installing the long-read noise removal tools...\n")
!sudo apt-get update
!sudo apt-get install porechop filtlong -y
print("\nTool installation completed successfully.")

### 13. Rationale 1: Coverage Depth Saturation

Combined, the paper's raw Illumina and Nanopore data provide **over 600x coverage**.
* For a small bacterial genome about 3.3 Mbp in size, 600x coverage means every single base of the genome is read 600 times over.
* For genome assembly algorithms, the gold-standard optimum depth for long reads is usually between **30x and 100x**. Going above this limit doesn't algorithmically add any new biological information; the data reaches saturation.

### 13. MAIN WORKFLOW: Step 1 - Trimming With Porechop
We apply the first noise removal stage set out by the paper by trimming leftover motor protein adapter residues and chimeric, erroneous reads from the ends of the Oxford Nanopore data.

In [ ]:
# 13. COLAB APPLICATION: Porechop Adapter Trimming
print("Step 1: TRIMMING the adapter noise in the OXFORD NANOPORE data with Porechop...\n")

!porechop -i /content/full_data_example/SRR13680735.fastq -o filtered_reads/trimmed_ONT.fastq

print("\nAdapters cleaned up successfully. The resulting file will be passed on to the filtering stage.")

### 13.2. MAIN WORKFLOW: Step 2 - Hybrid-Assisted Filtering With Filtlong
We're replicating the paper's methodology exactly. Using the `-1` and `-2` parameters, we feed the high-quality **Trimmomatic outputs** (Illumina reads) we just produced into the Filtlong algorithm.

The algorithm will use these short sequences as a template (16-mer) to test the quality of the Nanopore reads and select the best 90% slice.

In [ ]:
# 13.2. COLAB APPLICATION: Reference-Guided Filtering With Filtlong
print("Step 2: FILTERING the trimmed ONT reads with Filtlong...\n")
print("Paper strategy: Min length: 2000bp | Quality: 90% (Illumina-referenced) | Subset: 1.5 Gb\n")

# Bioinformatics note: the -1 and -2 flags pass in the Trimmomatic (Illumina) data as the short-read reference (guide).
!filtlong -1 filtered_reads/filtered_SRR13680736_1.fastq \
          -2 filtered_reads/filtered_SRR13680736_2.fastq \
          --min_length 2000 \
          --keep_percent 90 \
          --target_bases 1500000000 \
          filtered_reads/trimmed_ONT.fastq > filtered_reads/filtered_ONT.fastq

print("\nOxford Nanopore raw data filtered successfully, staying 100% faithful to the paper's parameters!")

## 14. Quality Control After Noise Removal (Post-QC)

We've done great work in the noise removal steps. But in bioinformatics it's not enough to just do something — you have to prove mathematically and visually that it worked. We'll use the NanoPlot tool again to re-measure the quality of our Nanopore data, now filtered to paper standards through Filtlong.

### 14.1. Re-Evaluating the Filtered Long Reads (Nanopore)
We now run the same NanoPlot analysis we did on the raw data on day one, this time on our clean file in the `filtered_reads` folder. We'll see the increase in "Read Length" and "Quality" in the resulting report with our own eyes.

In [ ]:
# 14. COLAB APPLICATION: NanoPlot After Noise Removal (Post-QC)
print("Running quality control (Post-QC) on the filtered OXFORD NANOPORE data...\n")

!NanoPlot --fastq filtered_reads/filtered_ONT.fastq -o post_qc_nanoplot/ --threads 2

print("\nPost-QC analysis complete! Please open the 'post_qc_nanoplot/' folder in the Colab file explorer and check the 'NanoPlot-report.html' file.")

### 15. Applying It: NanoPlot Analysis on the Noise-Cleared Long-Read Data

We trigger the NanoPlot analysis to examine the final health of our `filtered_ONT.fastq` file, which had its adapter noise trimmed by Porechop and was then filtered by Filtlong.

In [ ]:
# 15. COLAB APPLICATION: NanoPlot Report for the Filtered Long Reads
print("Running NanoPlot on the noise-cleared Oxford Nanopore data...\n")

# The '--threads 2' parameter runs the processor cores in parallel:
!NanoPlot --fastq filtered_reads/filtered_ONT.fastq -o re_qc_reports/nanoplot_re_qc --threads 2

print("\nLong-read Re-QC visual maps and statistical summaries generated successfully.")

### 16. Applying It: Producing the Final Re-QC Success Report With MultiQC

The MultiQC algorithm will go into the `re_qc_reports/` folder, automatically scan the FastQC and NanoPlot data we just produced, and combine it all into a single interactive report.

In [ ]:
# 16. COLAB APPLICATION: Combining the MultiQC Report
print("MultiQC is scanning all the new Re-QC logs and building the combined report...\n")

# The first parameter is the folder to scan, the '-o' parameter is the output folder:
!multiqc re_qc_reports/ -o re_qc_reports/

print("\nCongratulations! Your final interactive scorecard after noise removal and filtering is ready.")
print("Please download the newly created 'multiqc_report.html' file from the 're_qc_reports/' folder in the left panel to your computer.")
print("Open this new report side by side with the raw data report from the first lesson in your browser, and see your noise removal success for yourself!")

# BIF201 Applied Bioinformatics: Genomic Data Analyses (LIVE LESSON 16-17)

## LIVE LESSON 16-17: Aligning to the Real Reference Genome (Mapping) Workflow and Algorithmic Background

We're past the trimming and filtering stages; we now have Illumina and Oxford Nanopore reads with optimized quality scores, cleared of adapter artifacts. These reads are millions of tiny puzzle pieces of our *Gluconobacter cerinus FLW-1* genome project.

Our goal in this lesson is to find the original coordinates of our reads on the reference genome — in other words, to **align (map)** them onto the master map.

### 1. Algorithm Selection Strategy by Sequencing Technology
In the alignment step, we have to choose the most accurate mathematical approach based on read length and error profile:

* **Short Reads (Illumina) -> BWA-MEM:** short reads have a high accuracy rate but have trouble matching in repetitive regions of the genome. **BWA-MEM** compresses the reference genome in memory using the *Burrows-Wheeler Transform (BWT)* approach and finds the exact match points of short sequences in seconds.
* **Long Reads (Oxford Nanopore / PacBio) -> Minimap2:** reads that are tens of thousands of bases long can bridge structural variations and difficult genome regions, but carry raw deletion/insertion noise. Thanks to its split-read algorithm, **Minimap2** tolerates the noise flexibility of long reads beautifully.

### Expert Note: Alignment Algorithm Comparison Matrix

| Metric / Feature | Short-Read Alignment (BWA-MEM) | Long-Read Alignment (Minimap2) |
| :--- | :--- | :--- |
| **Core Algorithm** | Burrows-Wheeler Transform & Suffix Array | Seed-Chain-Align (super-fast approximate matching) |
| **Input Compatibility** | Illumina, Ion Torrent (short-read raw data) | Oxford Nanopore, PacBio (long-read raw data) |
| **Error Tolerance** | Low (focused on substitution errors) | High (flexible to deletion and insertion noise) |
| **Dominant Use Case** | Point mutation (SNP) and small indel detection | Large structural variants, de novo assembly bridges |

### 2. Installing the Alignment Tools and Obtaining the "Real" Reference Genome

Looking at our reference paper's data availability section, we see that the final reference genome for the *Gluconobacter cerinus FLW-1* project is stored in the NCBI database under accession number **JAFEJB010000000**.

We'll install the necessary packages on our virtual machine and download this original reference sequence directly from NCBI's servers into our workspace.

## 17: Aligning to the Real Reference Genome (Mapping) Workflow

We're past the trimming and filtering stages; we now have Illumina and Oxford Nanopore reads with optimized Phred quality scores, cleared of adapter artifacts. These reads are millions of tiny puzzle pieces of our *Gluconobacter cerinus FLW-1* genome project.

Our goal at this stage is to find the original coordinates of our reads on the reference genome — in other words, to **align (map)** them onto the master map.

### 17.1. Installing the Alignment Tools (BWA and Minimap2)
Based on the nature of the platforms, we're installing the BWA algorithm for short reads and the Minimap2 algorithm for long reads on our virtual machine.

In [ ]:
# 17. COLAB TERMINAL GUIDE: Installing the Tools
print("Installing the alignment tools (BWA and Minimap2) with the Linux package manager...\n")
!sudo apt-get update
!sudo apt-get install bwa minimap2 -y
print("\nTool installation completed successfully.")

### 18. Downloading the Reference Genome (Data Retrieval)
Looking at our reference paper's data availability section, we see that the project's final reference genome is stored in the NCBI database under accession number **JAFEJB010000000**. We're pulling this genome directly from NCBI's servers.

In [ ]:
# 18. COLAB TERMINAL GUIDE: Downloading the Original Reference Genome
print("Pulling the project's original reference genome (JAFEJB010000000) from NCBI servers...\n")
# Using the NCBI E-utilities API to download the sequence directly in FASTA format:
!curl -s "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=nuccore&id=JAFEJB010000000&rettype=fasta&retmode=text" > reference_genome.fasta
print("\nOur 'reference_genome.fasta' file was successfully added to the workspace.")